In [ ]:
#| default_exp pool

In [ ]:
#| hide
from fastcore.test import *
from nbdev.showdoc import *

One kernel per project, and a policy for how many stay alive at once.

`KernelPool` maps a key the host chooses, usually a notebook or a folder, to one live kernel. `RuntimeBroker` holds a ceiling across every pool in the process and decides what happens when it is reached.

`tests/test_pool.py` drives the pool against real kernels. The examples here use a stand-in with the attributes the pool and the broker read, so the admission and eviction rules can be shown with fixed numbers.

In [ ]:
#| export
from __future__ import annotations

In [ ]:
#| export
import asyncio, os, time

In [ ]:
#| export
from pathlib import Path

In [ ]:
#| export
from fastcore.all import L, first, ifnone

In [ ]:
#| export
from kunda.kernel import GatewayKernel, GatewayService, Kernel

In [ ]:
#| export
from kunda.kernel import KERNELS
from kunda.spec import KernelStartError, kernelspec_for

In [ ]:
#| export
#: How long a kernel may sit idle before the sweeper closes it. `$KUNDA_KERNEL_IDLE` overrides.
IDLE_SECONDS = 30*60
#: How often the sweeper looks. Never longer than `idle`, or a short idle never fires.
SWEEP_SECONDS = 60

`$KUNDA_KERNEL_IDLE` overrides `IDLE_SECONDS` for a pool that is not given an `idle` of its own. `0` turns the sweep off.

`SWEEP_SECONDS` bounds how often the sweep looks. `_sweep_loop` sleeps for the smaller of the two, so a pool with a 5 second idle is swept every 5 seconds rather than once a minute.

In [ ]:
#| export
class KernelLimit(RuntimeError):
    "The ceiling is reached. `candidate` is a kernel that can be offered to close, or None while all are busy."
    def __init__(self, msg, candidate=None, limit=0):
        super().__init__(msg)
        self.candidate, self.limit = candidate, limit

`KernelLimit` is what `admit` raises at the ceiling. It subclasses `RuntimeError`, so a host that already catches a failed start catches this too.

`candidate` describes the kernel the host may offer to close, and is None when every live kernel is busy. Nothing can be offered then, and the message says so instead.

In [ ]:
#| export
class RuntimeBroker:
    "A ceiling across every pool in this process. Never evicts a namespace somebody is using."
    def __init__(self, max_kernels=None, auto_manage=None):
        self.max_kernels = int(max_kernels or os.environ.get('KUNDA_MAX_KERNELS', 12))
        self.auto_manage = bool(ifnone(auto_manage,
            os.environ.get('KUNDA_KERNEL_AUTO', '').lower() in ('1', 'true', 'yes')))
        self.pools, self._lock, self._reserved = [], asyncio.Lock(), set()
    def register(self, pool):
        if pool not in self.pools: self.pools.append(pool)
        pool.broker = self
        return pool
    @property
    def live(self):
        return [(pool, key, kernel) for pool in self.pools for key, kernel in pool.kernels.items()
            if kernel.alive]
    def status(self):
        return {'live': len(self.live), 'limit': self.max_kernels,
            'runtimes': [{'key': str(key), 'pid': kernel.pid, 'kind': kernel.kernel_kind,
                'cwd': str(kernel.cwd or '')} for _, key, kernel in self.live]}
    def stalest(self):
        "The idle kernel that has waited longest, as `(pool, key, kernel)`. None while all are busy."
        idle = [row for row in self.live if not row[2].busy]
        return max(idle, key=lambda row: row[2].idle_for) if idle else None
    def _candidate(self):
        "What the limit dialog needs to name the kernel it is offering to close."
        if (row := self.stalest()) is None: return None
        _, key, k = row
        return {'key': str(key), 'name': Path(str(getattr(k, 'cwd', '') or key)).name,
                'cwd': str(getattr(k, 'cwd', '') or ''), 'pid': getattr(k, 'pid', None),
                'idle_for': int(k.idle_for)}
    async def admit(self, pool, key):
        token, evicted = (id(pool), key), None
        async with self._lock:
            if token in self._reserved or key in pool._starting: return
            if len(self.live) + len(self._reserved) >= self.max_kernels:
                row = self.stalest()
                if row is None: raise KernelLimit(
                    f'kernel limit reached ({self.max_kernels}) and every runtime is busy; stop one '
                    'or set KUNDA_MAX_KERNELS to a larger value', limit=self.max_kernels)
                if not self.auto_manage: raise KernelLimit(
                    f'kernel limit reached ({self.max_kernels}); close an idle runtime '
                    'or set KUNDA_MAX_KERNELS to a larger value',
                    candidate=self._candidate(), limit=self.max_kernels)
                evicted = self._candidate()
                vpool, vkey, _ = row
                await vpool.close(vkey)   # `close` never takes this lock, so holding it is safe
            self._reserved.add(token)
        return evicted
    async def release(self, pool, key):
        async with self._lock: self._reserved.discard((id(pool), key))

`RuntimeBroker` counts live kernels across every pool registered with it and refuses to go past `max_kernels`. `register` appends the pool and sets `pool.broker` on it, so either half finds the other. `$KUNDA_MAX_KERNELS` and `$KUNDA_KERNEL_AUTO` set the two defaults.

A start in flight counts against the ceiling. `admit` reserves `(pool, key)` before the kernel exists and `KernelPool.get` releases the reservation when the start finishes, so several panels asking at once cannot each pass a check only the first should pass.

The broker never closes a busy kernel. `stalest` considers only kernels with no cell running, and returns the one idle longest. With `auto_manage` off, reaching the ceiling raises `KernelLimit` naming that kernel and closes nothing. With `auto_manage` on, `admit` closes it and returns the same description, which the caller reports as what it took to make room.

The examples are below `KernelPool`, because the broker counts what a pool holds.

In [ ]:
#| export
class KernelPool:
    "Live kernels, keyed by an id the host assigns: usually a tab, a notebook or a folder."
    def __init__(self, port=8000, default_kernel='ipykernel', default_python=None, broker=None,
        transport='direct', gateway=None, idle=None, runner_for=None, known_kernels=None):
        #: `runner_for(lang)` gives a class for a language with no Jupyter kernel; None means there
        #: is none, and `installed_spec` raises naming what would install one.
        #: `known_kernels` is the host's own `{language: kernelspec}`, which wins where it answers.
        self.runner_for, self.known_kernels = runner_for, known_kernels or {}
        self.port, self.kernels, self._starting, self.broker = port, {}, {}, None
        self.idle = int(ifnone(idle, os.environ.get('KUNDA_KERNEL_IDLE', IDLE_SECONDS)))
        self._sweep = None
        self.transport = transport if transport in ('direct', 'gateway') else 'direct'
        self.gateway = gateway
        if broker is not None: broker.register(self)
        self.default_kernel = default_kernel if default_kernel in KERNELS else 'ipykernel'
        self.default_python = default_python
    def choose(self,
        kernel=None,
        python=False,
    ):
        "Set what the *next* kernel starts as. Running kernels are left alone."
        if kernel and kernel in KERNELS: self.default_kernel = kernel
        if python is not False: self.default_python = python
        return {'kernel': self.default_kernel, 'python': self.default_python}
    def _class_for(self, kw):
        """What runs this language: a Jupyter kernel where one is installed, else the host's own runner.

        The gateway transport carries Python. A language the host runs itself has no wire protocol to
        put through it, so it stays in this process whichever transport the workspace picked."""
        lang = kw.get('lang') or 'python'
        if lang == 'python': return GatewayKernel if self.transport == 'gateway' else Kernel
        if kernelspec_for(lang, self.known_kernels): return Kernel
        if self.runner_for and (r := self.runner_for(lang)) is not None: return r
        return Kernel                            # `installed_spec` raises, naming the install
    async def get(self, key, **kw):
        "The kernel for `key`, starting it once even when several browser panels ask together."
        self._sweeping()
        if (task := self._starting.get(key)) is not None: return await asyncio.shield(task)
        if (k := self.kernels.get(key)) is not None and k.alive: return k.touch()
        evicted = await self.broker.admit(self, key) if self.broker is not None else None
        async def start_one():
            kw.setdefault('kernel', self.default_kernel)
            kw.setdefault('python', self.default_python)
            cls, args = self._class_for(kw), dict(kw)
            if cls is GatewayKernel:
                if self.gateway is None: self.gateway = GatewayService().start()
                args['gateway'] = self.gateway
                args.pop('lang', None)          # the gateway carries Python, and takes no language
            # a host's own runner is keyed, because it has no connection file to be found by
            if getattr(cls, 'wants_key', False): args['key'] = key
            k = self.kernels[key] = cls(port=self.port, **args)
            k.evicted = evicted
            k.touch()
            try: return await k.start()
            except BaseException:
                if self.kernels.get(key) is k: self.kernels.pop(key, None)
                raise
        task = self._starting.get(key)
        if task is None: task = self._starting[key] = asyncio.create_task(start_one())
        try:
            return await asyncio.shield(task)
        finally:
            if task.done() and self._starting.get(key) is task:
                self._starting.pop(key, None)
                if self.broker is not None: await self.broker.release(self, key)
    async def reap(self):
        "Close what nobody has touched for `idle` seconds. A running cell is never taken."
        if self.idle <= 0: return []
        stale = [k for k, v in self.kernels.items()
                 if k not in self._starting and not v.busy and v.idle_for > self.idle]
        for k in stale: await self.close(k)
        return stale
    def _sweeping(self):
        "Start the idle sweep on first use: `get` is the only thing that ever adds a kernel."
        if self.idle > 0 and (self._sweep is None or self._sweep.done()):
            self._sweep = asyncio.create_task(self._sweep_loop())
    async def _sweep_loop(self):
        while True:
            await asyncio.sleep(min(SWEEP_SECONDS, max(1, self.idle)))
            try: await self.reap()
            except asyncio.CancelledError: raise
            except Exception: pass
    def peek(self, key):
        "A completed live kernel, never the object currently being started."
        if key in self._starting: return None
        k = self.kernels.get(key)
        return k if k is not None and k.alive else None
    async def close(self, key):
        if (task := self._starting.get(key)) is not None:
            try: await asyncio.shield(task)
            except BaseException: pass
        if (k := self.kernels.pop(key, None)) is not None: await k.shutdown()
    async def close_all(self):
        if self._sweep is not None: self._sweep.cancel(); self._sweep = None
        tasks = list(self._starting.values())
        if tasks: await asyncio.gather(*[asyncio.shield(task) for task in tasks], return_exceptions=True)
        await asyncio.gather(*[k.shutdown() for k in self.kernels.values()], return_exceptions=True)
        self.kernels.clear()

In [ ]:
#| hide
#| exec_doc
class Fake:
    "A kernel-shaped stand-in: what the pool and the broker read, and a shutdown that answers."
    alive, pid, kernel_kind = True, 4242, 'ipykernel'
    def __init__(self, cwd, idle_for=0., busy=False): self.cwd, self.idle_for, self.busy = cwd, idle_for, busy
    async def shutdown(self): self.alive = False

`KernelPool.get` is the only thing that starts a kernel, and it starts one per key however many callers ask together. The first caller stores the task in `_starting` and later callers await that same task. `peek` answers None for a key still starting, so a caller that must not wait never receives a half-started kernel. A start that raises removes the key, so the next caller starts a new kernel rather than being handed a dead one. `tests/test_pool.py` covers all of that against real processes.

`choose` sets what the next kernel starts as and leaves running kernels alone. A `kernel` name outside `KERNELS` changes nothing. A `python` of `False` means the argument was not passed. That leaves `None` free to mean this interpreter.

In [ ]:
#| exec_doc
p = KernelPool()
p.choose(kernel='ipymini', python='/proj/.venv/bin/python')

In [ ]:
#| hide
test_eq(p.choose(kernel='nonsense')['kernel'], 'ipymini')     # an unknown name changes nothing
test_eq(p.choose()['python'], '/proj/.venv/bin/python')       # and not passing it is not clearing it
test_is(p.choose(python=None)['python'], None)                # passing None is

`_class_for` picks what runs a language. The gateway transport carries Python. A language with an installed kernelspec, and a language the host runs itself through `runner_for`, both stay in this process whichever transport the workspace picked, because neither has a wire protocol to put through the gateway. A `transport` that is neither `direct` nor `gateway` is read as `direct`.

An unknown language with no `runner_for` resolves to `Kernel`, whose start raises naming what would install a kernelspec for it.

In [ ]:
#| exec_doc
gw = KernelPool(transport='gateway')
(gw._class_for({'lang': 'python'}).__name__, gw._class_for({'lang': 'rust'}).__name__,
 KernelPool(transport='carrier-pigeon').transport)

`reap` closes what nobody has touched for `idle` seconds and returns the keys it took. A kernel running a cell is never taken, however long ago it was last touched, and neither is one that is still starting. `idle` of 0 turns the sweep off and `reap` takes nothing.

The sweep starts on the first `get` and runs until `close_all` cancels it.

In [ ]:
#| exec_doc
pool = KernelPool(idle=60)
pool.kernels.update(stale=Fake('/proj/stale', idle_for=90), fresh=Fake('/proj/fresh', idle_for=5),
                    working=Fake('/proj/working', idle_for=900, busy=True))
await pool.reap(), list(pool.kernels)

In [ ]:
#| hide
test_eq(pool.kernels['fresh'].alive, True)
pool.idle = 0
test_eq(await pool.reap(), [])                # switched off, and the same kernels are still stale
test_is(pool.peek('working'), pool.kernels['working'])
test_is(pool.peek('gone'), None)
await pool.close_all()
test_eq(pool.kernels, {})

The examples below fill a pool with stand-ins rather than starting kernels. Two kernels, a ceiling of two, and one of them busy.

In [ ]:
#| exec_doc
b = RuntimeBroker(max_kernels=2, auto_manage=False)
pool = KernelPool(idle=0, broker=b)
pool.kernels.update(one=Fake('/proj/one', idle_for=90), two=Fake('/proj/two', busy=True))
b.status()

`status` is what a runtime panel lists. `_candidate` describes one of those kernels for the dialog that asks whether to close it, and carries the name of the folder it runs in as well as the key.

In [ ]:
#| exec_doc
b.stalest()[1], b._candidate()

At the ceiling with an idle kernel to offer, `admit` raises and closes nothing. The host shows the message and the candidate, and the person decides.

In [ ]:
#| exec_doc
try: await b.admit(pool, 'three')
except KernelLimit as e: print(e); print(e.candidate['name'], e.limit)

With every kernel busy there is nothing to offer, so `candidate` is None and the message names the ceiling on its own.

In [ ]:
#| exec_doc
pool.kernels['one'].busy = True
try: await b.admit(pool, 'three')
except KernelLimit as e: print(e); print(e.candidate, b.stalest())

With `auto_manage` on, the stalest kernel is closed and `admit` returns its description. The new kernel's `evicted` attribute carries it. The host reports from there which namespace it took to make room.

In [ ]:
#| exec_doc
auto = RuntimeBroker(max_kernels=1, auto_manage=True)
p2 = KernelPool(idle=0, broker=auto)
p2.kernels['one'] = Fake('/proj/one', idle_for=90)
evicted = await auto.admit(p2, 'two')
evicted['key'], list(p2.kernels), auto.status()['live']

In [ ]:
#| hide
test_eq(auto._reserved, {(id(p2), 'two')})    # nothing is alive yet, and the slot is held anyway
await auto.release(p2, 'two')
test_eq(auto._reserved, set())
test_is(await auto.admit(p2, 'two'), None)    # room again, so nothing was evicted to make it
await auto.release(p2, 'two')
test_eq(b.pools, [pool]); test_is(pool.broker, b)
await pool.close_all(); await p2.close_all()